# Recurrent Neural Network (RNN) for Text Generation

**Name:** Pradeep Manikandan D  
**Roll Number:** `YOUR_ROLL_NUMBER`  

## Aim
To implement a Recurrent Neural Network (RNN) for text generation and evaluate its ability to generate meaningful text by predicting the next word in a sequence.

## Software Requirements
- Python
- TensorFlow / Keras
- Google Colab
- GitHub
- NumPy
- Matplotlib

## Objectives
1. Understand the architecture and working of RNNs.
2. Perform text preprocessing for sequence modeling.
3. Generate input sequences for next-word prediction.
4. Build and train an RNN text-generation model.
5. Generate text by predicting the next word.
6. Evaluate the trained RNN using generated text.
7. Analyze how effectively the RNN learns sequential patterns.


## PART A — Dataset Preparation

The Tiny Shakespeare dataset is loaded from Hugging Face. The text is then cleaned, tokenized, converted into integer sequences, and padded so that all training samples have the same length.

In [ ]:
# PART A: Imports and student details

import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf

from datasets import load_dataset
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences

print("Name: Pradeep Manikandan D")
print("Roll Number: YOUR_ROLL_NUMBER")
print("TensorFlow version:", tf.__version__)


In [ ]:
# PART A: Load Tiny Shakespeare from Hugging Face

# This dataset configuration provides the Tiny Shakespeare text.
# If the dataset repository used by your lab changes, only DATASET_NAME
# needs to be changed.

DATASET_NAME = "Trelis/tiny-shakespeare"

dataset = load_dataset(DATASET_NAME)

print(dataset)
print("\nAvailable splits:", list(dataset.keys()))


In [ ]:
# Explore the dataset and identify the text column

split_name = list(dataset.keys())[0]
data = dataset[split_name]

print(data)
print("\nColumn names:", data.column_names)
print("\nFirst sample:")
print(data[0])


In [ ]:
# Extract text and display a sample

TEXT_COLUMN = "text"

if TEXT_COLUMN not in data.column_names:
    # Automatically find a likely text column if the expected name is unavailable.
    text_candidates = [c for c in data.column_names if data.features[c].dtype == "string"]
    if not text_candidates:
        raise ValueError("No text column was found in the selected dataset.")
    TEXT_COLUMN = text_candidates[0]

text = "\n".join(str(x) for x in data[TEXT_COLUMN])

print("Text column:", TEXT_COLUMN)
print("Number of characters:", len(text))
print("\nFirst 1000 characters:\n")
print(text[:1000])


In [ ]:
# Convert text to lowercase

text = text.lower()

print(text[:500])


In [ ]:
# Tokenization and vocabulary creation

tokenizer = Tokenizer(
    filters='!"#$%&()*+,-./:;<=>?@[\\]^_`{|}~\\t\\n',
    lower=True,
    oov_token="<OOV>"
)

tokenizer.fit_on_texts([text])

total_words = len(tokenizer.word_index) + 1

print("Vocabulary size:", total_words)
print("\nFirst 20 vocabulary entries:")
print(list(tokenizer.word_index.items())[:20])


In [ ]:
# Generate next-word prediction sequences

token_list = tokenizer.texts_to_sequences([text])[0]

# Keep the sequence length manageable for a classroom experiment.
SEQ_LENGTH = 10

input_sequences = []

for i in range(SEQ_LENGTH, len(token_list)):
    sequence = token_list[i-SEQ_LENGTH:i+1]
    input_sequences.append(sequence)

input_sequences = np.array(input_sequences)

X = input_sequences[:, :-1]
y = input_sequences[:, -1]

print("Input shape before padding:", X.shape)
print("Target shape:", y.shape)
print("Example input:", X[0])
print("Example target:", y[0])


In [ ]:
# Apply sequence padding

X = pad_sequences(
    X,
    maxlen=SEQ_LENGTH,
    padding="pre",
    truncating="pre"
)

print("Input shape after padding:", X.shape)
print("Example padded sequence:", X[0])


In [ ]:
# Train-validation split

split_index = int(0.9 * len(X))

X_train = X[:split_index]
y_train = y[:split_index]

X_val = X[split_index:]
y_val = y[split_index:]

print("Training samples:", len(X_train))
print("Validation samples:", len(X_val))
print("Training input shape:", X_train.shape)
print("Validation input shape:", X_val.shape)


**Screenshot checkpoint — Part A:** Run the above cells and capture screenshots showing the student details, dataset exploration, vocabulary, sequence generation, padding, and train/validation split.

## PART B — Implementing the RNN Model

The model contains:
- An **Embedding** layer for dense word representations.
- A **SimpleRNN** layer for learning sequential dependencies.
- A **Dense + Softmax** output layer for next-word prediction.

In [ ]:
# PART B: Build the Simple RNN model

from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, SimpleRNN, Dense

EMBEDDING_DIM = 64
RNN_UNITS = 128

model = Sequential([
    Embedding(input_dim=total_words, output_dim=EMBEDDING_DIM, input_length=SEQ_LENGTH),
    SimpleRNN(RNN_UNITS),
    Dense(total_words, activation="softmax")
])

model.compile(
    optimizer="adam",
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

model.summary()


**Screenshot checkpoint — Part B:** Capture the model summary showing the Embedding, SimpleRNN, and Dense layers.

## PART C — Training and Evaluation

The RNN is trained on the prepared sequences. Training and validation accuracy/loss are recorded and visualized against epochs.

In [ ]:
# PART C: Train the RNN

EPOCHS = 10
BATCH_SIZE = 256

history = model.fit(
    X_train,
    y_train,
    validation_data=(X_val, y_val),
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    verbose=1
)


In [ ]:
# Record final training and validation metrics

final_train_accuracy = history.history["accuracy"][-1]
final_val_accuracy = history.history["val_accuracy"][-1]
final_train_loss = history.history["loss"][-1]
final_val_loss = history.history["val_loss"][-1]

print(f"Training Accuracy   : {final_train_accuracy:.4f}")
print(f"Validation Accuracy : {final_val_accuracy:.4f}")
print(f"Training Loss       : {final_train_loss:.4f}")
print(f"Validation Loss     : {final_val_loss:.4f}")


In [ ]:
# Accuracy vs Epoch

plt.figure(figsize=(8, 5))
plt.plot(history.history["accuracy"], label="Training Accuracy")
plt.plot(history.history["val_accuracy"], label="Validation Accuracy")
plt.title("Accuracy vs Epoch")
plt.xlabel("Epoch")
plt.ylabel("Accuracy")
plt.legend()
plt.grid(True)
plt.show()


In [ ]:
# Loss vs Epoch

plt.figure(figsize=(8, 5))
plt.plot(history.history["loss"], label="Training Loss")
plt.plot(history.history["val_loss"], label="Validation Loss")
plt.title("Loss vs Epoch")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.legend()
plt.grid(True)
plt.show()


In [ ]:
# Summary table of final performance

print("FINAL PERFORMANCE")
print("-" * 35)
print(f"Training Accuracy   : {final_train_accuracy:.4f}")
print(f"Validation Accuracy : {final_val_accuracy:.4f}")
print(f"Training Loss       : {final_train_loss:.4f}")
print(f"Validation Loss     : {final_val_loss:.4f}")


**Screenshot checkpoint — Part C:** Capture the training output, final metrics, Accuracy vs Epoch graph, and Loss vs Epoch graph.

## PART D — Text Generation

The trained RNN predicts one word at a time. Each predicted word is appended to the current sequence and the process is repeated to generate a complete text sample.

In [ ]:
# PART D: Reverse vocabulary for converting predicted indices into words

index_to_word = {index: word for word, index in tokenizer.word_index.items()}

def generate_text(seed_text, next_words=30, temperature=0.8):
    """Generate text using the trained RNN."""
    generated = seed_text.lower()

    for _ in range(next_words):
        token_list = tokenizer.texts_to_sequences([generated])[0]
        token_list = pad_sequences(
            [token_list],
            maxlen=SEQ_LENGTH,
            padding="pre",
            truncating="pre"
        )

        predictions = model.predict(token_list, verbose=0)[0]

        # Temperature controls randomness.
        predictions = np.asarray(predictions).astype("float64")
        predictions = np.log(predictions + 1e-8) / temperature
        probabilities = np.exp(predictions) / np.sum(np.exp(predictions))

        predicted_id = np.random.choice(len(probabilities), p=probabilities)

        next_word = index_to_word.get(predicted_id, "")
        if not next_word or next_word == "<OOV>":
            continue

        generated += " " + next_word

    return generated


In [ ]:
# Generate multiple samples using different seed inputs

seed_inputs = [
    "to be or not",
    "the king is",
    "my lord and",
    "what is the"
]

for seed in seed_inputs:
    print("=" * 80)
    print("Seed:", seed)
    print("Generated text:")
    print(generate_text(seed, next_words=30, temperature=0.8))
    print()


In [ ]:
# Compare generated samples

samples = {}

for seed in seed_inputs:
    samples[seed] = generate_text(seed, next_words=40, temperature=0.8)

for seed, sample in samples.items():
    print(f"Seed: {seed}")
    print(sample)
    print("\n")


### Observation

The generated text should demonstrate that the RNN has learned some sequential and linguistic patterns from the Tiny Shakespeare dataset. However, a Simple RNN may produce grammatically imperfect, repetitive, or contextually inconsistent sentences, especially when the generated sequence becomes long. Different seed inputs can lead to different generated outputs because the model predicts subsequent words based on the preceding context.

**Screenshot checkpoint — Part D:** Capture the generated text for multiple seed inputs and the observations.

Finally, save the notebook and upload it to GitHub as required by the laboratory instructions.

# Expected Outcome

The RNN successfully learns sequential patterns from the Tiny Shakespeare text and performs next-word prediction. The experiment demonstrates the complete workflow of text preprocessing, tokenization, sequence generation, padding, RNN model construction, training, validation, visualization of accuracy and loss, and text generation from different seed inputs.

# Viva / Record Note

**Student Name:** Pradeep Manikandan D  
**Roll Number:** YOUR_ROLL_NUMBER  

Replace `YOUR_ROLL_NUMBER` with your actual roll number before submission.